# Supplementary Note 8 — gaze position at target onset

A control for the obvious alternative reading of Figure 1. If the preview lets
observers move their eyes toward the upcoming face BEFORE the search display
appears, they start closer to the target and reach it sooner, and the benefit is
oculomotor rather than a matter of scene guidance. Earlier controls could not
address this: they were all restricted to eye movements *following* target onset.

Eye position at onset is read from the fixation that was in progress when the
display appeared. Recording begins at display onset, so that fixation's start
time is clipped to the event and its mean position reflects the oculomotor state
established BEFORE the target — saccade latencies of at least ~150 ms rule out
any target-driven change within it. The estimate is event-based, not a
continuous trace: the 500 ms fixation-dot interval was not sampled, so this is
the only pre-target position available.

Two distances are computed from that position, in degrees of visual angle:
distance from the central fixation point, which asks whether the preview moves
the eye at all, and distance from the face centroid, which asks whether the
movement is toward the target. Only the second is the confound. A displacement
unrelated to the face buys no head start.

The analysis runs on all successful trials of both experiments. Distances are
pooled within participant by the MEDIAN, since a handful of trials begin with
the eye far off centre; latency regressions are on the raw millisecond scale
rather than logs, because the bound below multiplies degrees by ms/deg and needs
an additive scale.

In [ ]:
import sys
from pathlib import Path

# The notebooks live next to the code, so the package is importable without
# installing anything. Nothing outside this release is referenced.
sys.path.insert(0, str(Path.cwd().parent / "code"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from facescenes import config as cfg
from facescenes import data as D
from facescenes import behaviour as B
from facescenes import encoding as E
from facescenes import onset_gaze as G
from facescenes import plotting as P
from facescenes import stats as S

P.use_style()
print("stimuli    :", len(D.stimulus_names()))
for t in cfg.TASKS:
    print(f"{t:14s}: {len(D.participant_ids(t))} participants, "
          f"{len(D.analysable_images(t))} images")

## S7 — is the eye displaced, and is it displaced toward the face?

Left panel: distance of onset gaze from the screen centre. Right panel: distance
from the face centroid. Each line is a participant.

In [ ]:
def onset_panel(ax, df, column, ylabel, result):
    """Per-participant median distance in the two conditions."""
    a, b = B.participant_means(df, column, agg="median")
    for i in range(len(a)):
        ax.plot([0, 1], [a[i], b[i]], color="grey", lw=.6, alpha=.5, zorder=1)
    box = ax.boxplot([a, b], positions=[0, 1], widths=.5, patch_artist=True,
                     showfliers=False, zorder=2, medianprops=dict(color="k", lw=2))
    for patch, cond in zip(box["boxes"], cfg.CONDITIONS):
        patch.set_facecolor(P.condition_colours()[cond]); patch.set_edgecolor("k")
    ax.set_xticks([0, 1])
    ax.set_xticklabels([cfg.COND_LABEL[c] for c in cfg.CONDITIONS],
                       fontsize=P.FONT - 14)
    ax.set_ylabel(ylabel, fontsize=P.FONT - 14)
    ax.set_ylim(0, np.nanmax([a, b]) * 1.15)
    ax.text(.5, .99, P.significance(result["p"]), ha="center", va="top",
            transform=ax.transAxes, fontsize=P.FONT - 12)
    P.clean(ax)


TABLES = {}      # reused by the cells below, so the table is built once per task

for task, fname in (("facedetection", "suppnote8a_onset_gaze_face_detection.png"),
                    ("movingwindow", "suppnote8b_onset_gaze_moving_window.png")):
    TABLES[task] = df = G.onset_gaze_table(task)
    disp = G.displacement(task, df)

    fig, axes = plt.subplots(1, 2, figsize=(9, 5.5))
    onset_panel(axes[0], df, "d_centre_deg",
                "Distance from screen\ncentre at onset (deg)", disp["d_centre_deg"])
    onset_panel(axes[1], df, "d_face_deg",
                "Distance from face\ncentroid at onset (deg)", disp["d_face_deg"])
    P.title(fig, cfg.TASK_LABEL[task], fontsize=P.FONT - 6, y=1.02)
    fig.tight_layout()
    src = pd.concat(
        [B.participant_table(df, c, agg="median").rename(
            columns=lambda k, c=c: f"{k}_median_{c}") for c in ("d_centre_deg", "d_face_deg")],
        axis=1).reset_index()
    P.save(fig, fname, src); plt.show()

    dropped = len(D.trials(task)) - len(df)
    print(f"{cfg.TASK_LABEL[task]} — {len(df)} trials, "
          f"{dropped} dropped with no qualifying onset fixation "
          f"({100 * dropped / (len(df) + dropped):.1f} %)")
    for col, label in (("d_centre_deg", "from screen centre"),
                       ("d_face_deg", "to face centroid")):
        r = disp[col]
        print(f"  {label:19s} {r['mean_a']:5.2f} vs {r['mean_b']:5.2f} deg   "
              f"diff {r['diff']:+.2f} deg, 95% CI "
              f"[{r['diff_ci'][0]:+.2f}, {r['diff_ci'][1]:+.2f}], "
              f"t({r['n']-1}) = {r['t']:.2f}, p = {r['p']:.3g}, dz = {r['dz']:+.2f}")

## Can the displacement account for the benefit?

Three questions in order.

**Does onset position matter at all?** The positive control. Latency is regressed
on preview condition and both distances, separately within each participant, and
the gaze-to-face slope is tested across participants. It must be positive, or the
confound has no mechanism to work through.

**How much of the benefit could it buy?** The most favourable end of the
confidence interval for the condition difference in gaze-to-face distance, times
that slope. Deliberately generous: it grants the confound the largest
displacement toward the face the data will tolerate.

**Does the benefit survive adjustment?** The preview coefficient with and without
the distances as covariates. All four estimates are regression coefficients from
the same within-participant model, so they are comparable with each other —
setting a median-based effect against a regression coefficient makes adjustment
appear to change an estimate when it has not.

In [ ]:
for task in cfg.TASKS:
    df = TABLES[task]
    print(cfg.TASK_LABEL[task])

    for col, label in (("d_face_deg", "gaze-to-face"),
                       ("d_centre_deg", "gaze-to-centre")):
        s = G.distance_slope(task, col, df)
        print(f"  latency on {label:15s} {s['mean']:+8.2f} ms/deg   "
              f"t({s['n']-1}) = {s['t']:.2f}, p = {s['p']:.3g}")

    b = G.orienting_bound(task, df)
    print(f"  maximum attributable to orienting: {b['ms']:.1f} ms "
          f"({b['degrees']:+.2f} deg x {b['ms_per_degree']:.1f} ms/deg)")

    for covs, label in (((), "unadjusted"),
                        (("d_face_deg",), "+ gaze-to-face"),
                        (("d_centre_deg",), "+ gaze-to-centre"),
                        (("d_face_deg", "d_centre_deg"), "+ both")):
        e = G.preview_effect(task, covs, df)
        print(f"  preview effect {label:17s} {e['mean']:+9.2f} ms, 95% CI "
              f"[{e['ci'][0]:+.1f}, {e['ci'][1]:+.1f}], p = {e['p']:.3g}, "
              f"dz = {e['dz']:+.2f}")
    print()

## Trials reached in a single fixation

The face detection task only, as in Supplementary Figure 2. On these trials the
whole of detection latency is one saccade launched from the onset position, so
the exchange rate between distance and latency is not diluted by the extra
fixations of longer searches. It is the strictest form of the reviewer's own
account, and the version the earlier draft of this control reported for
Experiment 1. The moving-window task is not shown: only a small minority of its
trials end this way.

Reported here as a robustness check rather than as the primary analysis, so that
both experiments are analysed on the same trials.

In [ ]:
task = "facedetection"
hits = TABLES[task][TABLES[task]["n_fixations"] == 1]
print(f"{cfg.TASK_LABEL[task]}: {len(hits)}/{len(TABLES[task])} trials "
      f"({100 * len(hits) / len(TABLES[task]):.1f} %) ended on the first fixation\n")

r = G.displacement(task, hits)["d_face_deg"]
print(f"  distance to face centroid  diff {r['diff']:+.2f} deg, 95% CI "
      f"[{r['diff_ci'][0]:+.2f}, {r['diff_ci'][1]:+.2f}], p = {r['p']:.3g}")
s = G.distance_slope(task, "d_face_deg", hits)
print(f"  latency on gaze-to-face    {s['mean']:+.2f} ms/deg, p = {s['p']:.3g}")
b = G.orienting_bound(task, hits)
print(f"  maximum attributable to orienting: {b['ms']:.1f} ms")
for covs, label in (((), "unadjusted"),
                    (("d_face_deg", "d_centre_deg"), "+ both")):
    e = G.preview_effect(task, covs, hits)
    print(f"  preview effect {label:12s} {e['mean']:+7.2f} ms, 95% CI "
          f"[{e['ci'][0]:+.1f}, {e['ci'][1]:+.1f}], p = {e['p']:.3g}")